In [1]:
import pandas as pd
import requests

In [35]:
url = "https://odds.p.rapidapi.com/v4/sports/upcoming/odds"

querystring = {"regions":"us","oddsFormat":"decimal","markets":"h2h,spreads","dateFormat":"iso"}

headers = {
	"x-rapidapi-key": "",
	"x-rapidapi-host": "odds.p.rapidapi.com",
	"Content-Type": "application/json"
}

response = requests.get(url, headers=headers, params=querystring)

print(response.json())

{'message': 'Invalid API key. Go to https://docs.rapidapi.com/docs/keys for more info.'}


In [28]:
data = response.json()

temp_df = pd.DataFrame([{
    
    # match info
    'sport_key': match.get('sport_key'),
    'commence_time': match.get('commence_time'),
    'home_team': match.get('home_team'),
    'away_team': match.get('away_team'),
    
    # 🔥 best odds (from all bookmakers)
    'home_team_odds': min([
        outcome['price']
        for bookmaker in match.get('bookmakers', [])
        for market in bookmaker.get('markets', [])
        if market.get('key') == 'h2h'
        for outcome in market.get('outcomes', [])
        if outcome.get('name') == match.get('home_team')
    ], default=None),
    
    'away_team_odds': min([
        outcome['price']
        for bookmaker in match.get('bookmakers', [])
        for market in bookmaker.get('markets', [])
        if market.get('key') == 'h2h'
        for outcome in market.get('outcomes', [])
        if outcome.get('name') == match.get('away_team')
    ], default=None),
    
   
    'home_point': next((
        outcome.get('point')
        for bookmaker in match.get('bookmakers', [])
        for market in bookmaker.get('markets', [])
        if market.get('key') == 'spreads'
        for outcome in market.get('outcomes', [])
        if outcome.get('name') == match.get('home_team')
    ), None),
    
    'away_point': next((
        outcome.get('point')
        for bookmaker in match.get('bookmakers', [])
        for market in bookmaker.get('markets', [])
        if market.get('key') == 'spreads'
        for outcome in market.get('outcomes', [])
        if outcome.get('name') == match.get('away_team')
    ), None),

} for match in data])

print(temp_df.head())

               sport_key         commence_time                     home_team  \
0           baseball_npb  2026-04-12T04:00:00Z  Hokkaido Nippon-Ham Fighters   
1           baseball_npb  2026-04-12T04:00:38Z           Saitama Seibu Lions   
2           baseball_npb  2026-04-12T04:01:00Z  Tohoku Rakuten Golden Eagles   
3           baseball_npb  2026-04-12T04:30:00Z              Chunichi Dragons   
4  soccer_korea_kleague1  2026-04-12T05:00:00Z               Daejeon Citizen   

                away_team  home_team_odds  away_team_odds  home_point  \
0  Fukuoka SoftBank Hawks            3.15            1.24         1.5   
1     Chiba Lotte Marines            1.69            1.83         1.5   
2          Orix Buffaloes            1.01           34.00        -3.5   
3          Hanshin Tigers           15.00            1.00         2.5   
4              Gangwon FC          151.00            1.02         NaN   

   away_point  
0        -1.5  
1        -1.5  
2         3.5  
3        -2.5  


In [29]:
temp_df

,sport_key,commence_time,home_team,away_team,home_team_odds,away_team_odds,home_point,away_point
0,baseball_npb,2026-04-12T04:00:00Z,Hokkaido Nippon-Ham Fighters,Fukuoka SoftBank Hawks,3.15,1.24,1.50,-1.50
1,baseball_npb,2026-04-12T04:00:38Z,Saitama Seibu Lions,Chiba Lotte Marines,1.69,1.83,1.50,-1.50
2,baseball_npb,2026-04-12T04:01:00Z,Tohoku Rakuten Golden Eagles,Orix Buffaloes,1.01,34.00,-3.50,3.50
3,baseball_npb,2026-04-12T04:30:00Z,Chunichi Dragons,Hanshin Tigers,15.00,1.00,2.50,-2.50
4,soccer_korea_kleague1,2026-04-12T05:00:00Z,Daejeon Citizen,Gangwon FC,151.00,1.02,NaN,NaN
5,soccer_australia_aleague,2026-04-12T05:00:00Z,Melbourne City,Wellington Phoenix FC,1.00,901.00,NaN,NaN
6,soccer_japan_j_league,2026-04-12T05:00:00Z,Urawa Red Diamonds,Tokyo Verdy,18.00,23.00,NaN,NaN
7,baseball_kbo,2026-04-12T05:00:28Z,Hanwha Eagles,Kia Tigers,2.55,1.19,1.50,-1.50
8,baseball_kbo,2026-04-12T05:00:41Z,LG Twins,SSG Landers,1.01,9.75,-4.50,4.50
9,baseball_kbo,2026-04-12T05:00:46Z,KT Wiz,Doosan Bears,1.02,7.00,-3.00,3.00


In [30]:
df = pd.DataFrame()

In [31]:

df = pd.concat([df, temp_df], ignore_index=True)


In [32]:
df.head()

,sport_key,commence_time,home_team,away_team,home_team_odds,away_team_odds,home_point,away_point
0,baseball_npb,2026-04-12T04:00:00Z,Hokkaido Nippon-Ham Fighters,Fukuoka SoftBank Hawks,3.15,1.24,1.5,-1.5
1,baseball_npb,2026-04-12T04:00:38Z,Saitama Seibu Lions,Chiba Lotte Marines,1.69,1.83,1.5,-1.5
2,baseball_npb,2026-04-12T04:01:00Z,Tohoku Rakuten Golden Eagles,Orix Buffaloes,1.01,34.00,-3.5,3.5
3,baseball_npb,2026-04-12T04:30:00Z,Chunichi Dragons,Hanshin Tigers,15.00,1.00,2.5,-2.5
4,soccer_korea_kleague1,2026-04-12T05:00:00Z,Daejeon Citizen,Gangwon FC,151.00,1.02,NaN,NaN


In [33]:
df.shape

(23, 8)

In [34]:
df.to_csv('theoddsAPI.csv')